# Stage 1 — Case Base: Data Acquisition & Preprocessing

**Project:** Case-Based Reasoning (CBR) untuk Analisis Putusan Pengadilan Indonesia  
**Tahap:** 1 dari 4 — Membangun Case Base  
**Environment:** VSCode + Jupyter (Lokal)  

---

## Overview

Notebook ini membangun **case base** untuk sistem CBR dengan cara:

1. Memuat file PDF putusan pengadilan dari folder dataset lokal.
2. Melakukan extraction teks polos menggunakan `pdfplumber`.
3. Melakukan cleaning pada teks hasil extraction (watermark, whitespace, format teks).
4. Menyimpan hasil cleaning teks ke dalam struktur folder project.
5. Menghasilkan log preprocessing dan laporan validation.

**Keputusan Desain Penting:**
- Tanpa stemming, lemmatization, atau stopword removal — teks hukum memerlukan preservasi konteks yang utuh.
- Pembersihan watermark yang disesuaikan dengan format PDF Mahkamah Agung.
- Pipeline dapat diskalakan untuk 30+ dokumen tanpa perubahan kode.

---

## Daftar Isi

1. [Environment Setup](#1.-Environment-Setup)
2. [Configure Paths](#2.-Configure-Paths)
3. [Create Project Structure](#3.-Create-Project-Structure)
4. [Scan PDF Files](#4.-Scan-PDF-Files)
5. [PDF Text Extraction](#5.-PDF-Text-Extraction)
6. [Text Cleaning](#6.-Text-Cleaning)
7. [Save Cleaned Text](#7.-Save-Cleaned-Text)
8. [Logging](#8.-Logging)
9. [Validation](#9.-Validation)
10. [Final Summary](#10.-Final-Summary)

**Lampiran (Appendices):**
- [A. Future Improvements](#A.-Future-Improvements)
- [B. Validation Checklist](#B.-Validation-Checklist)

---

# 1. Environment Setup

Instalasi dependencies yang diperlukan dan import seluruh library untuk pipeline.

**Library yang digunakan:**
- `pdfplumber` — PDF text extraction (layout-aware, tanpa OCR).
- `pandas` — Manipulasi data dan pembuatan tabel summary.
- `tqdm` — Progress bar untuk batch processing.
- `pathlib`, `os`, `glob`, `re` — File handling dan pemrosesan teks (stdlib).

In [ ]:
# Install dependencies (run once per environment)

%pip install pdfplumber tqdm pandas -q

In [ ]:
# Import libraries

import os
import re
import glob
import logging
import warnings
from pathlib import Path
from datetime import datetime

import pdfplumber
import pandas as pd
from tqdm import tqdm

# Suppress noisy warnings from pdfplumber internals
warnings.filterwarnings('ignore')

print("All libraries imported successfully")
print(f"Session started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---

# 2. Configure Paths

Mengatur path project root dan dataset pada filesystem lokal.

Path project root akan di-resolve secara otomatis berdasarkan lokasi notebook ini.  
Folder dataset (`datasets uu-ite/`) berisi file PDF putusan pengadilan.

In [ ]:
from pathlib import Path
import os

# CONFIGURATION — Local path setup

PROJECT_ROOT = Path("../").resolve()

DATASET_DIR = PROJECT_ROOT / "datasets" / "uu-ite"

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOG_DIR = PROJECT_ROOT / "logs"

# Verify paths


print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset folder: {DATASET_DIR}")
print()

if DATASET_DIR.exists():

    contents = os.listdir(DATASET_DIR)

    pdf_count = sum(
        1 for f in contents
        if f.lower().endswith(".pdf")
    )

    print(" Dataset folder found")
    print(f"   Total items: {len(contents)}")
    print(f"   PDF files: {pdf_count}")

else:

    print(f"❌ Dataset folder NOT found:")
    print(DATASET_DIR)

---

# 3. Create Project Structure

Membuat seluruh direktori project yang diperlukan secara otomatis.

```text
project-cbr/
├── datasets uu-ite/  # PDF putusan pengadilan (input)
├── data/
│   ├── raw/          # File teks bersih (output Stage 1)
│   ├── processed/    # Representasi kasus terstruktur (Stage 2)
│   ├── eval/         # Dataset evaluasi (Stage 4)
│   └── results/      # Summary pipeline dan laporan
├── notebooks/        # Jupyter notebook
└── logs/             # Log pemrosesan
```

In [ ]:
# Create project directory structure

DIRECTORIES = [
    PROJECT_ROOT / 'data' / 'raw',
    PROJECT_ROOT / 'data' / 'processed',
    PROJECT_ROOT / 'data' / 'eval',
    PROJECT_ROOT / 'data' / 'results',
    PROJECT_ROOT / 'notebooks',
    PROJECT_ROOT / 'logs',
]

print("📁 Creating project structure...\n")

for dir_path in DIRECTORIES:
    dir_path.mkdir(parents=True, exist_ok=True)
    # Show relative path for readability
    relative = dir_path.relative_to(PROJECT_ROOT)
    print(f"  📂 {relative}/")

print(f"\n Project structure created at: {PROJECT_ROOT}")

---

# 4. Scan PDF Files

Mendeteksi secara otomatis seluruh file PDF di dalam folder dataset lokal.  
Tidak ada nama file yang di-hardcode — pipeline secara dinamis menemukan seluruh file `.pdf`.

Ini mendukung skalabilitas untuk 30+ dokumen tanpa adanya perubahan kode.

In [ ]:
# Auto-scan all PDF files in the dataset folder

def scan_pdf_files(folder_path):
    pdf_files = []
    for ext in ('*.pdf', '*.PDF'):
        pdf_files.extend(glob.glob(os.path.join(folder_path, ext)))

    # Remove duplicates (in case filesystem is case-insensitive) and sort
    pdf_files = sorted(set(pdf_files))
    return pdf_files


# Run the scan
pdf_files = scan_pdf_files(DATASET_DIR)

if pdf_files:
    print(f"📄 Found {len(pdf_files)} PDF file(s):\n")
    for i, pdf_path in enumerate(pdf_files, 1):
        file_size_kb = os.path.getsize(pdf_path) / 1024
        print(f"  {i:3d}. {os.path.basename(pdf_path):50s} ({file_size_kb:,.1f} KB)")
else:
    print(f"No PDF files found in: {DATASET_DIR}")
    print("Please check the folder path and try again.")

---

# 5. PDF Text Extraction

Mengekstrak teks dari setiap PDF menggunakan **pdfplumber**, yang menyediakan layout-aware text extraction tanpa memerlukan proses OCR.

**Mengapa pdfplumber?**
- Menangani tata letak multi-kolom yang umum ditemukan pada putusan pengadilan Indonesia.
- Mempertahankan urutan pembacaan teks dengan lebih baik dibanding `pdfminer` biasa.
- Tidak memiliki dependency eksternal (Tesseract, Poppler, dll.).

**Batasan (Limitations):**
- Tidak dapat mengekstrak teks dari PDF hasil scan/hanya gambar (memerlukan OCR fallback).
- Beberapa PDF dengan pemformatan kompleks mungkin menghasilkan output teks yang berantakan.

In [ ]:
# Define the PDF text extraction function

def extract_text_from_pdf(pdf_path):
    """
    Extract all text from a PDF file using pdfplumber.

    Iterates over every page, extracts text, and joins them
    with double-newlines as page separators.

    Args:
        pdf_path (str): Absolute path to the PDF file.

    Returns:
        dict: Extraction result with keys:
            - filename (str): Original PDF filename
            - filepath (str): Full path to the PDF
            - raw_text (str): Extracted raw text
            - page_count (int): Number of pages in the PDF
            - status (str): 'success', 'warning', or 'failed'
            - error (str|None): Error message if any
    """
    result = {
        'filename': os.path.basename(pdf_path),
        'filepath': pdf_path,
        'raw_text': '',
        'page_count': 0,
        'status': 'success',
        'error': None,
    }

    try:
        with pdfplumber.open(pdf_path) as pdf:
            result['page_count'] = len(pdf.pages)
            pages_text = []

            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    pages_text.append(text)

            result['raw_text'] = '\n\n'.join(pages_text)

            # Flag if no text was extracted (likely a scanned PDF)
            if not result['raw_text'].strip():
                result['status'] = 'warning'
                result['error'] = 'No text extracted — possible scanned/image-only PDF'

    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)

    return result

In [ ]:
# Run extraction on all detected PDFs

print(" Extracting text from PDF files...\n")

extraction_results = []

for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    result = extract_text_from_pdf(pdf_path)
    extraction_results.append(result)

    # Status indicator
    icons = {'success': '✅', 'warning': '⚠️', 'failed': '❌'}
    icon = icons.get(result['status'], '❓')
    char_count = len(result['raw_text'])

    print(f"  {icon} {result['filename']}")
    print(f"     Pages: {result['page_count']}, Characters: {char_count:,}")
    if result['error']:
        print(f"     ⚠ {result['error']}")

# Quick tally
n_ok = sum(1 for r in extraction_results if r['status'] == 'success')
n_warn = sum(1 for r in extraction_results if r['status'] == 'warning')
n_fail = sum(1 for r in extraction_results if r['status'] == 'failed')

print(f"\n Extraction complete — {n_ok} success, {n_warn} warning(s), {n_fail} failure(s)")

---

# 6. Text Cleaning

Clean teks hasil extraction dengan tetap menjaga makna hukum di dalamnya.

**Tahap cleaning yang diterapkan:**
1. Menghapus watermark yang diketahui (disclaimer Mahkamah Agung, URL, marker halaman) — regex block-level.
2. Mengubah teks menjadi lowercase.
3. Menghapus karakter non-printable / control characters.
4. Menormalisasi horizontal whitespace (tab, multiple spaces menjadi single space).
5. Menghilangkan trailing whitespace pada setiap baris teks.
6. Menghapus baris disclaimer/watermark yang tersisa (substring-based, lebih robust terhadap garbling) dan fragmen watermark vertikal (`ma ubl`).
7. Menghapus karakter noise tunggal di awal/akhir baris yang bocor dari watermark sidebar vertikal (dengan safeguard untuk referensi hukum seperti "huruf a" dan kata berspasi seperti "p u t u s a n").
8. Mereduksi baris kosong berlebih (3+ baris kosong berturut-turut menjadi 2 baris kosong).

**Mengapa digabung jadi satu fungsi?**
Watermark pada dokumen putusan Mahkamah Agung muncul dalam dua bentuk: blok disclaimer multi-baris yang bisa ditangkap regex, dan fragmen/noise per-baris yang baru terlihat *setelah* lowercasing & whitespace normalization. Karena itu deteksi substring & noise inline dijalankan sebagai langkah lanjutan dalam fungsi `clean_text` yang sama sehingga seluruh pembersihan teks terjadi dalam satu pass yang konsisten.

**Sengaja TIDAK diterapkan:**
- Stemming : karena dapat merusak terminologi hukum (misal: "memutuskan" vs "putusan").
- Lemmatization : alasan yang sama.
- Stopword removal : kata seperti "dan", "atau", "tidak" memiliki interpretasi hukum yang sangat penting.


In [ ]:
# Define watermark patterns and the text cleaning function

# Common watermarks/headers/footers found in Indonesian court decision PDFs
# (particularly those downloaded from putusan.mahkamahagung.go.id)
WATERMARK_PATTERNS = [
    # Mahkamah Agung directory header
    r'Direktori\s+Putusan\s+Mahkamah\s+Agung\s+Republik\s+Indonesia',
    # Website watermark
    r'putusan\.mahkamahagung\.go\.id',
    # Standard disclaimer block (multi-line, captured generously)
    r'Disclaimer\s*\n.*?Kepaniteraan\s+Mahkamah\s+Agung.*?(?=\n\n|\Z)',
    # Page number markers: "Halaman X dari Y ..."
    r'Halaman\s+\d+\s+dari\s+\d+[^\n]*',
    # Sometimes the full court header repeats on every page
    r'Mahkamah\s+Agung\s+Republik\s+Indonesia\s*\n\s*Mahkamah\s+Agung\s+Republik\s+Indonesia',
]

# Substring markers — sisa watermark/disclaimer yang mengalami garbling
WATERMARK_MARKERS = [
    "selalu mencantumkan informasi",
    "paling kini dan akurat",
    "pelayanan publik, transparansi",
    "gsi peradilan",
    "menemukan inakurasi",
    "mahkamahagung.go.id",
    "021-384 3348",
]

# Baris yang persis sama dengan fragmen watermark vertikal (setelah strip + lowercase)
WATERMARK_EXACT = {"ma ubl", "ma ubl."}

# Deteksi inline noise: karakter tunggal di awal/akhir baris
LEADING_NOISE_RE = re.compile(r"^([a-z]) (?=[a-z]{2,}|\d)")
TRAILING_NOISE_RE = re.compile(r"\s+[a-z]$")

# Kata referensi hukum yang sah diikuti huruf tunggal (misal: "huruf a") tidak dihapus.
LEGAL_REF_WORDS = {"huruf", "sub", "poin", "butir", "angka", "lampiran", "golongan"}


def is_watermark_line(line_text):
    stripped = line_text.strip().lower()
    if not stripped:
        return False
    if stripped in WATERMARK_EXACT:
        return True
    return any(marker in stripped for marker in WATERMARK_MARKERS)


# Vertical sidebar watermark sering bocor sebagai SATU baris per karakter
SINGLE_CHAR_LINE_RE = re.compile(r"^[a-z]$")


def is_single_char_bleed(line_text):
    return bool(SINGLE_CHAR_LINE_RE.match(line_text.strip()))


def has_spaced_word_suffix(text):
    return bool(re.search(r"(?:[a-z] ){2,}[a-z]$", text.strip()))


def _strip_inline_noise(line):
    stripped = LEADING_NOISE_RE.sub('', line)

    m = TRAILING_NOISE_RE.search(stripped)
    if m and not has_spaced_word_suffix(stripped):
        words = stripped.rsplit(maxsplit=2)
        if not (len(words) >= 2 and words[-2].lower() in LEGAL_REF_WORDS):
            stripped = TRAILING_NOISE_RE.sub('', stripped)

    return stripped


def clean_text(text):
    if not text or not text.strip():
        return ''

    cleaned = text

    # Step 1: Remove known block-level watermarks, headers, and footers
    for pattern in WATERMARK_PATTERNS:
        cleaned = re.sub(pattern, '', cleaned, flags=re.IGNORECASE | re.DOTALL)

    # Step 2: Convert to lowercase
    cleaned = cleaned.lower()

    # Step 3: Remove non-printable/control characters
    # Keep: printable ASCII, standard whitespace, and extended Latin characters
    # (covers Indonesian text which uses standard Latin alphabet)
    cleaned = re.sub(r'[^\x20-\x7E\n\r\t\u00C0-\u024F]', '', cleaned)

    # Step 4: Normalize horizontal whitespace (tabs + multiple spaces -> single space)
    cleaned = re.sub(r'[^\S\n]+', ' ', cleaned)

    # Step 5: Strip leading/trailing whitespace on each line
    lines = [line.strip() for line in cleaned.split('\n')]

    # Step 6-7: Remove residual watermark lines + strip inline noise, line by line
    fixed_lines = []
    for line in lines:
        if not line:
            fixed_lines.append('')
            continue
        if is_watermark_line(line):
            continue
        if is_single_char_bleed(line):
            continue
        fixed_lines.append(_strip_inline_noise(line))

    cleaned = '\n'.join(fixed_lines)

    # Step 8: Collapse 3+ consecutive blank lines into a single blank line
    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)

    # Final strip
    cleaned = cleaned.strip()

    return cleaned


print(" Cleaning function defined")
print(f"   {len(WATERMARK_PATTERNS)} block-level watermark pattern(s) loaded")
print(f"   {len(WATERMARK_MARKERS)} substring watermark marker(s) loaded")
print(f"   single-character vertical bleed lines will be removed")


In [ ]:
# Apply cleaning to all extracted texts

print(" Cleaning extracted texts...\n")

for result in tqdm(extraction_results, desc="Cleaning"):
    if result['status'] in ('success', 'warning') and result['raw_text']:
        result['cleaned_text'] = clean_text(result['raw_text'])
        result['raw_char_count'] = len(result['raw_text'])
        result['cleaned_char_count'] = len(result['cleaned_text'])

        # Calculate size reduction
        if result['raw_char_count'] > 0:
            reduction_pct = (1 - result['cleaned_char_count'] / result['raw_char_count']) * 100
        else:
            reduction_pct = 0.0

        print(f"  [DONE] {result['filename']}")
        print(f"     {result['raw_char_count']:,} → {result['cleaned_char_count']:,} chars "
              f"({reduction_pct:.1f}% reduction)")
    else:
        result['cleaned_text'] = ''
        result['raw_char_count'] = 0
        result['cleaned_char_count'] = 0
        print(f"{result['filename']}: skipped (no text available)")

print(f"\n Cleaning complete for {len(extraction_results)} file(s)")

---

# 7. Save Cleaned Text

Menyimpan setiap teks bersih hasil cleaning ke dalam direktori `/data/raw/` sebagai file `.txt`.

**Aturan penamaan file:** `case_001_<sanitized_original_name>.txt`  
Ini mempertahankan traceability ke file PDF asli sekaligus menambahkan ID berurutan (*sequential ID*).

In [ ]:
# Save cleaned text files to /data/raw/
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'raw'

print(" Saving cleaned text files...\n")

saved_count = 0
skipped_count = 0

for i, result in enumerate(extraction_results, 1):
    # Build output filename: case_001_originalname.txt
    original_stem = Path(result['filename']).stem
    safe_name = re.sub(r'[^\w\-]', '_', original_stem)  # sanitize
    output_filename = f"case_{i:03d}_{safe_name}.txt"
    output_path = OUTPUT_DIR / output_filename

    # Store metadata for later use in logging/validation
    result['case_id'] = f"case_{i:03d}"
    result['output_filename'] = output_filename
    result['output_path'] = str(output_path)

    if result.get('cleaned_text'):
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(result['cleaned_text'])
        saved_count += 1
        print(f"  [DONE] {output_filename} ({result['cleaned_char_count']:,} chars)")
    else:
        skipped_count += 1
        print(f"  [SKIPPED]  {output_filename} (empty text)")

print(f"\n Output directory: {OUTPUT_DIR}")
print(f"Saved: {saved_count} file(s)")
if skipped_count > 0:
    print(f" Skipped: {skipped_count} file(s)")

---

# 8. Logging

Menghasilkan file log terstruktur pada `/logs/cleaning.log` untuk mendokumentasikan seluruh proses preprocessing.

**Detail log untuk setiap file:**
- Nama file asli.
- Status pemrosesan (success/warning/failed).
- Jumlah halaman.
- Jumlah karakter (raw dan cleaned).
- Nama file output.
- Pesan error (jika ada).

In [ ]:
LOG_PATH = PROJECT_ROOT / 'logs' / 'cleaning.log'
timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

with open(LOG_PATH, 'w', encoding='utf-8') as log_file:
    # Header
    log_file.write(f"{'=' * 72}\n")
    log_file.write(f"  CBR Case Base — Preprocessing Log\n")
    log_file.write(f"  Generated: {timestamp}\n")
    log_file.write(f"  Source folder: {DATASET_DIR}\n")
    log_file.write(f"  Total files: {len(extraction_results)}\n")
    log_file.write(f"{'=' * 72}\n\n")

    # Per-file details
    for result in extraction_results:
        log_file.write(f"File:           {result['filename']}\n")
        log_file.write(f"  Case ID:      {result.get('case_id', 'N/A')}\n")
        log_file.write(f"  Status:       {result['status']}\n")
        log_file.write(f"  Pages:        {result['page_count']}\n")
        log_file.write(f"  Raw chars:    {result.get('raw_char_count', 0):,}\n")
        log_file.write(f"  Cleaned chars:{result.get('cleaned_char_count', 0):,}\n")
        log_file.write(f"  Output file:  {result.get('output_filename', 'N/A')}\n")
        if result.get('error'):
            log_file.write(f"  Error:        {result['error']}\n")
        log_file.write(f"\n")

    # Summary footer
    total = len(extraction_results)
    n_success = sum(1 for r in extraction_results if r['status'] == 'success')
    n_warnings = sum(1 for r in extraction_results if r['status'] == 'warning')
    n_failed = sum(1 for r in extraction_results if r['status'] == 'failed')
    total_pages = sum(r['page_count'] for r in extraction_results)
    total_chars = sum(r.get('cleaned_char_count', 0) for r in extraction_results)

    log_file.write(f"{'=' * 72}\n")
    log_file.write(f"  SUMMARY\n")
    log_file.write(f"{'=' * 72}\n")
    log_file.write(f"  Total files processed:  {total}\n")
    log_file.write(f"  Successful:             {n_success}\n")
    log_file.write(f"  Warnings:               {n_warnings}\n")
    log_file.write(f"  Failed:                 {n_failed}\n")
    log_file.write(f"  Total pages:            {total_pages:,}\n")
    log_file.write(f"  Total cleaned chars:    {total_chars:,}\n")
    log_file.write(f"{'=' * 72}\n")

print(f"Log saved to: {LOG_PATH}")
print(f"\n{'—' * 60}")
print("Log contents:\n")

# Display the log
with open(LOG_PATH, 'r', encoding='utf-8') as f:
    print(f.read())

---

# 9. Validation

Menjalankan quality check otomatis pada hasil extraction:

| Check | Kondisi | Severity |
|-------|-----------|----------|
| Extraction failure | `status == 'failed'` | 🔴 Critical |
| Empty text | `cleaned_char_count == 0` | 🔴 Critical |
| Suspiciously short | `cleaned_char_count < 3000` | 🟡 Warning |

Langkah ini juga menampilkan preview contoh teks dari file pertama yang sukses diekstrak.

In [ ]:
# Run validation checks

print(" Running validation checks...\n")

# Threshold: files shorter than this are flagged as suspicious.
MIN_CHAR_THRESHOLD = 3000

issues_critical = []
issues_warning = []

for result in extraction_results:
    filename = result['filename']
    char_count = result.get('cleaned_char_count', 0)

    # Check 1: Extraction failures
    if result['status'] == 'failed':
        issues_critical.append(
            f"FAILED: {filename} — {result.get('error', 'Unknown error')}"
        )

    # Check 2: Empty output
    elif char_count == 0:
        issues_critical.append(
            f"EMPTY: {filename} — No text extracted (0 characters)"
        )

    # Check 3: Suspiciously short text
    elif char_count < MIN_CHAR_THRESHOLD:
        issues_warning.append(
            f"SHORT: {filename} — Only {char_count:,} chars "
            f"(threshold: {MIN_CHAR_THRESHOLD:,})"
        )

# Report issues
if issues_critical:
    print("CRITICAL issues:\n")
    for issue in issues_critical:
        print(f"  {issue}")
    print()

if issues_warning:
    print("Warnings:\n")
    for issue in issues_warning:
        print(f"  {issue}")
    print()

if not issues_critical and not issues_warning:
    print(" All files passed validation — no issues detected!\n")

# Check 4: Sisa artefak watermark/disclaimer pada teks yang sudah dibersihkan
remaining_wm = 0
remaining_bleed = 0
for result in extraction_results:
    for line in result.get('cleaned_text', '').split('\n'):
        if is_watermark_line(line):
            remaining_wm += 1
        if is_single_char_bleed(line):
            remaining_bleed += 1

if remaining_wm == 0:
    print("Tidak ada artefak watermark/disclaimer tersisa pada teks yang sudah dibersihkan\n")
else:
    print(f"Masih ditemukan {remaining_wm} baris artefak watermark pada teks yang sudah dibersihkan\n")

if remaining_bleed == 0:
    print("Tidak ada baris vertical watermark bleed (satu karakter per baris) tersisa\n")
else:
    print(f"Masih ditemukan {remaining_bleed} baris vertical watermark bleed tersisa\n")


# Sample text preview

PREVIEW_LENGTH = 1000

print(f"{'—' * 60}")
print("📖 Sample text preview (first successful file):\n")

previewed = False
for result in extraction_results:
    if result.get('cleaned_text'):
        preview = result['cleaned_text'][:PREVIEW_LENGTH]
        print(f"File: {result['filename']}")
        print(f"Case ID: {result.get('case_id', 'N/A')}")
        print(f"{'—' * 40}")
        print(preview)
        print(f"{'—' * 40}")
        remaining = result['cleaned_char_count'] - PREVIEW_LENGTH
        if remaining > 0:
            print(f"(showing first {PREVIEW_LENGTH:,} of {result['cleaned_char_count']:,} chars, "
                  f"{remaining:,} more)")
        previewed = True
        break

if not previewed:
    print("  No text available for preview.")

---

# 10. Final Summary

Melihat statistik hasil akhir pipeline dan melakukan penilaian kesiapan data untuk masuk ke Stage 2.

In [ ]:
# Final summary statistics

total = len(extraction_results)
success = sum(1 for r in extraction_results if r['status'] == 'success')
warnings_count = sum(1 for r in extraction_results if r['status'] == 'warning')
failed = sum(1 for r in extraction_results if r['status'] == 'failed')
total_pages = sum(r['page_count'] for r in extraction_results)
total_raw_chars = sum(r.get('raw_char_count', 0) for r in extraction_results)
total_cleaned_chars = sum(r.get('cleaned_char_count', 0) for r in extraction_results)

print("=" * 60)
print("STAGE 1 — CASE BASE PREPROCESSING SUMMARY")
print("=" * 60)
print(f"""
  Total PDFs processed:       {total}
  Successful extractions:      {success}
  Warnings:                   {warnings_count}
  Failed extractions:          {failed}

  Total pages processed:       {total_pages:,}
  Total raw characters:        {total_raw_chars:,}
  Total cleaned characters:    {total_cleaned_chars:,}

  Cleaned text output:         {PROJECT_ROOT / 'data' / 'raw'}
  Preprocessing log:           {PROJECT_ROOT / 'logs' / 'cleaning.log'}
  Summary CSV:                 {PROJECT_ROOT / 'data' / 'results' / 'preprocessing_summary.csv'}
""")
print("=" * 60)

if failed == 0 and success > 0:
    print("\n Stage 1 complete! The case base is ready for Stage 2 (Case Representation).")
elif failed > 0:
    print(f"\n {failed} file(s) failed extraction. Review the log and fix before Stage 2.")
else:
    print("\n No files were successfully processed. Check the dataset path and PDF files.")

In [ ]:
# Generate and save a summary DataFrame

summary_df = pd.DataFrame([
    {
        'Case ID': r.get('case_id', f"case_{i:03d}"),
        'Original File': r['filename'],
        'Status': r['status'],
        'Pages': r['page_count'],
        'Raw Chars': r.get('raw_char_count', 0),
        'Cleaned Chars': r.get('cleaned_char_count', 0),
        'Output File': r.get('output_filename', 'N/A'),
        'Error': r.get('error', ''),
    }
    for i, r in enumerate(extraction_results, 1)
])

# Save to CSV
summary_csv_path = PROJECT_ROOT / 'data' / 'results' / 'preprocessing_summary.csv'
summary_df.to_csv(summary_csv_path, index=False, encoding='utf-8')
print(f" Summary saved to: {summary_csv_path}\n")

# Display the table
summary_df

# Appendices

## A. Future Improvements

The following enhancements may be considered in future iterations of the preprocessing pipeline:

* OCR fallback for scanned PDF documents that do not contain extractable text.
* Section-based segmentation for legal document structures such as:
  * Identitas Perkara
  * Dakwaan
  * Pertimbangan Hukum
  * Amar Putusan
* Duplicate document detection using content hashing.
* Additional metadata extraction from court decision documents.
* Improved handling of extraction artifacts from complex PDF layouts.

These improvements are intentionally deferred because they are not required for Stage 1 preprocessing.

---

## B. Validation Checklist

Stage 1 preprocessing can be considered complete when the following conditions are satisfied:

### Extraction

* [x] All PDF files are successfully detected.
* [x] Text extraction completes without critical errors.
* [x] No empty output files are generated.

### Cleaning

* [x] Watermarks and repetitive document artifacts are removed.
* [x] Legal content remains intact after cleaning.
* [x] Text is normalized and readable.

### Output

* [x] Cleaned text files are saved in `data/raw/`.
* [x] File naming follows the project convention.
* [x] Logs and preprocessing summaries are generated successfully.

### Readiness

- [x] Extracted documents are suitable for metadata extraction.
- [x] Extracted documents are suitable for case representation.
- [x] The pipeline runs end-to-end without manual intervention.
---

*Stage 1 (Case Base & Preprocessing) is considered complete when all validation items above have been reviewed and verified.*
